# Train Lora + LLM Judge + RAG Evaluation 

**Nguyen tac cua notebook nay:**
- **1 co `USE_RAG` DUY NHAT** dung chung cho Buoc 5 (run_evaluation) va
  Buoc 6 (LLM Judge) -- de dam bao evaluate LUON khop dieu kien luc train
  (train khong RAG thi evaluate cung khong RAG, va nguoc lai).
- **Chat (Buoc 7)** la doc lap -- tu chon bat/tat RAG rieng cho tung cau
  hoi bang lenh `/rag on` / `/rag off`, KHONG anh huong den co USE_RAG
  chung o tren.

**QUAN TRONG -- doc truoc khi chay:**
1. Runtime -> Disconnect and delete runtime (don sach session cu).
2. Runtime -> Change runtime type -> GPU (T4).
3. Chay tuan tu tu tren xuong, KHONG bo qua cell nao.
4. Sau Buoc 3 (cai dat), co cell yeu cau RESTART SESSION -- bat buoc.
5. Buoc 5 (diagnostic) neu co dong "[FAIL]" thi DUNG LAI xu ly truoc.


# Phần 1: Chuẩn bị cài đặt môi trường

## Buoc 0: Kiem tra GPU

In [ ]:
!nvidia-smi


In [ ]:
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n[FAIL] KHONG CO GPU. Doi Runtime type -> GPU (T4), roi Disconnect and delete runtime.")
else:
    print("[OK] GPU san sang.")


## Buoc 1: Clone project SACH + PATCH ngay (truoc khi cai dat)

Patch `trust_remote_code=True` cho nomic-embed-text phai lam NGAY SAU KHI
CLONE, truoc ca buoc cai dat/diagnostic -- neu de sau se bi FAIL o buoc
kiem tra rag_bridge.

In [ ]:
import shutil, os

PROJECT_DIR = "/kaggle/working/MockProject_062026_NhomAI/Training"

import sys
sys.path.insert(0, PROJECT_DIR)
!git clone --branch tuanphat --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git

%cd /kaggle/working/MockProject_062026_NhomAI/Training

!git clone --branch han --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git _RAG_

In [ ]:
import os
assert os.path.exists("pipeline/evaluate.py"), "Khong tim thay pipeline/evaluate.py -- clone loi"
assert os.path.exists("_RAG_/Embbeding_RAG"), "Khong tim thay _RAG_/Embbeding_RAG -- clone loi"
print("[OK] Clone dung vi tri, cau truc thu muc hop le.")

In [ ]:
# PATCH: nomic-embed-text-v1.5 can trust_remote_code=True (kien truc custom)
NOMIC_DIR = "/kaggle/working/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5"

!sed -i 's/model_name=embedding_model_name$/model_name=embedding_model_name, trust_remote_code=True/' \
  "{NOMIC_DIR}/test_vector_db/retrieval.py"
!sed -i 's/model_name=EMBEDDING_MODEL_NAME$/model_name=EMBEDDING_MODEL_NAME, trust_remote_code=True/' \
  "{NOMIC_DIR}/embedding_storage.py"

# Xac nhan patch da vao dung file
!grep -n "trust_remote_code" "{NOMIC_DIR}/test_vector_db/retrieval.py" || echo "[FAIL] Patch chua vao retrieval.py"


## Buoc 3: Cai dat package (1 LAN DUY NHAT, du het trong 1 lenh)

In [ ]:
%cd /kaggle/working/MockProject_062026_NhomAI/Training

!pip install --upgrade pip -q
!pip install \
    -r requirements_ver1.txt \
    -r _RAG_/Embbeding_RAG/requirements.txt \
    -q
!pip install -U "opentelemetry-api==1.44.0" "opentelemetry-sdk==1.44.0" -q


## >>> RESTART SESSION <<<

In [1]:
print("Restart session Completely.")


Restart session Completely.


## Buoc 4: DAT CO USE_RAG CHUNG (dung cho ca Buoc 5 va Buoc 6)

Doi gia tri True/False O DAY DUY NHAT -- dam bao evaluate luon khop dieu
kien luc train (vd: train KHONG dung RAG -> de False; neu ban co train
lai VOI RAG thi doi thanh True).

In [3]:
import os
os.chdir("/kaggle/working/MockProject_062026_NhomAI/Training")

import sys
sys.path.insert(0, "/kaggle/working/MockProject_062026_NhomAI/Training")

# ============================================================
# >>> DOI GIA TRI NAY CHO KHOP VOI LUC TRAIN <<<
TRAIN_EVAL_USE_RAG = False   # True neu model duoc train/muon eval CO RAG
# ============================================================

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MEDQUAD_RAG_DIR"] = "/kaggle/working/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5"
os.environ["MEDQUAD_USE_RAG"] = "1" if TRAIN_EVAL_USE_RAG else "0"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

print(f"[OK] Da set MEDQUAD_USE_RAG = {os.environ['MEDQUAD_USE_RAG']} (dung cho Buoc 5 + Buoc 6)")

[OK] Da set MEDQUAD_USE_RAG = 0 (dung cho Buoc 5 + Buoc 6)


## Buoc 5: DIAGNOSTIC -- kiem tra du dieu kien truoc khi chay that

In [4]:
import torch
print(("[OK]" if torch.cuda.is_available() else "[FAIL]"), "GPU:", torch.cuda.is_available())


[OK] GPU: True


In [5]:
try:
    import opentelemetry.sdk.environment_variables as ev
    _ = ev.OTEL_LOGRECORD_ATTRIBUTE_COUNT_LIMIT
    print("[OK] opentelemetry OK. File:", ev.__file__)
except Exception as e:
    print("[FAIL] opentelemetry loi:", e)


[OK] opentelemetry OK. File: /usr/local/lib/python3.12/dist-packages/opentelemetry/sdk/environment_variables/__init__.py


In [6]:
try:
    import chromadb
    print("[OK] chromadb OK, version:", chromadb.__version__)
except Exception as e:
    print("[FAIL] chromadb loi:", e)


[OK] chromadb OK, version: 1.5.9


In [7]:
try:
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    print("[OK] ragas OK.")
except Exception as e:
    print("[FAIL] ragas loi:", e)


[OK] ragas OK.


In [8]:
try:
    from src.config import USE_RAG, ADAPTER_DIR, PREDICTIONS_CSV
    print("[OK] src.config OK. USE_RAG =", USE_RAG)
    print("ADAPTER_DIR:", ADAPTER_DIR, "| ton tai:", ADAPTER_DIR.exists())
    print("PREDICTIONS_CSV:", PREDICTIONS_CSV)
except Exception as e:
    print("[FAIL] src.config loi:", e)


[OK] src.config OK. USE_RAG = False
ADAPTER_DIR: /kaggle/working/MockProject_062026_NhomAI/Training/output/output_model | ton tai: True
PREDICTIONS_CSV: /kaggle/working/MockProject_062026_NhomAI/Training/output/evaluation_results.csv


In [9]:
from src.config import USE_RAG
import os

if USE_RAG:
    rag_dir = os.environ["MEDQUAD_RAG_DIR"]
    chroma_path = os.path.join(rag_dir, "chroma_db", "chroma.sqlite3")
    if os.path.exists(chroma_path):
        print("[OK] chroma_db ton tai tai:", chroma_path)
    else:
        print(f"[FAIL] KHONG tim thay {chroma_path} -- can build chroma_db truoc.")
else:
    print("[SKIP] USE_RAG=False -- khong can check chroma_db.")


[SKIP] USE_RAG=False -- khong can check chroma_db.


In [10]:
from src.config import USE_RAG

if USE_RAG:
    try:
        from src.rag_bridge import get_context_with_similarity
        r = get_context_with_similarity("What are the symptoms of diabetes?", top_k=2)
        if r["raw_contexts"]:
            print(f"[OK] rag_bridge retrieve OK -- {len(r['raw_contexts'])} chunks, rag_used={r['rag_used']}")
            print("Preview:", r["raw_contexts"][0][:200])
        else:
            print("[FAIL] rag_bridge chay khong loi nhung RONG -- kiem tra lai chroma_db co du lieu chua.")
    except Exception as e:
        print("[FAIL] rag_bridge loi:", type(e).__name__, "-", e)
else:
    print("[SKIP] USE_RAG=False -- khong can test rag_bridge.")


[SKIP] USE_RAG=False -- khong can test rag_bridge.


In [11]:
from src.config import ADAPTER_DIR
if ADAPTER_DIR.exists() and any(ADAPTER_DIR.iterdir()):
    print("[OK] Tim thay model da train tai:", ADAPTER_DIR)
else:
    print(f"[FAIL] Chua co model tai {ADAPTER_DIR} -- can train truoc hoac tai model len dung vi tri nay.")


[OK] Tim thay model da train tai: /kaggle/working/MockProject_062026_NhomAI/Training/output/output_model


In [12]:
print("=" * 60)
print("Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.")
print("=" * 60)


Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.


# Phần 2: Train

## Bước 1: Build train.jsonl từ final_train_dataset.json

In [13]:
from pipeline import build_train_dataset
build_train_dataset.main()

Total samples (raw)   : 37172
Sample limit áp dụng  : None
Hợp lệ (validated)    : 37172
Bỏ qua (thiếu Q/A)    : 0
USE_RAG               : False
----------------------------------------
Train : 26020 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/train.jsonl
Val   : 5575 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/val.jsonl
Test  : 5577 -> /kaggle/working/MockProject_062026_NhomAI/Training/output/test.jsonl


## Bước 2: Train Lora

### Bước 2.1: Training (Nếu đã có model -> Bỏ qua)

In [ ]:
from pipeline import train
train.main()

### Bước 2.2: Nếu đã có model

In [14]:
from pipeline import run_evaluation
run_evaluation.main()   # full tap test, ghi đè từ đầu
# run_evaluation.main(resume=True)  # dùng nếu bị ngắt giữa chừng


Đang tải tập test (thô)...
Đang load model đã train từ /kaggle/working/MockProject_062026_NhomAI/Training/output/output_model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Load xong.
Số câu hỏi test: 5577 | Sẽ chạy: 5577
USE_RAG=False -> chạy Q&A thuần, không có ngữ cảnh (như cũ).
[EVAL] Bắt đầu MỚI -- sẽ ghi đè /kaggle/working/MockProject_062026_NhomAI/Training/output/evaluation_results.csv nếu đã tồn tại.
[EVAL] Mục tiêu: 5577 mẫu.
[EVAL] Đã inference 5/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 10/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 15/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 20/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 25/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 30/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 35/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 40/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 45/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 50/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 55/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 60/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 65/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 70/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 75/5577 mẫu... (đã lưu CSV)
[EVAL] Đã infer

KeyboardInterrupt: 

## Buoc 3: LLM Judge (RAGAs) -- van dung DUNG USE_RAG cua Phan 1

In [15]:
import getpass
os.environ["MEDQUAD_JUDGE_API_KEY"] = getpass.getpass("Nhập API Key")
os.environ["MEDQUAD_JUDGE_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["MEDQUAD_JUDGE_API_MODEL"] = "llama-3.1-8b-instant"

import importlib
import src.config
importlib.reload(src.config)   # doc lai JUDGE_API_KEY vua nhap

from pipeline import evaluate
importlib.reload(evaluate)
evaluate.main()


Nhập API Key ········


Kết quả đánh giá sẽ được lưu (append theo batch) vào: /kaggle/working/MockProject_062026_NhomAI/Training/output/ragas_scores.csv
Kết quả đánh giá sẽ được lưu (append theo batch) vào: /kaggle/working/MockProject_062026_NhomAI/Training/output/ragas_scores.csv
Đang đọc CSV dự đoán (đã sinh sẵn từ bước train, gồm ROUGE/BLEU)...
Số mẫu: 330
Đang khởi tạo model giám khảo (tách biệt model vừa train)...
Bật JSON mode cho giám khảo (giảm lỗi parse với model nhỏ). Nếu thấy TOÀN BỘ câu bị lỗi/NaN sau khi bật, set MEDQUAD_JUDGE_JSON_MODE=0 và chạy lại để tắt JSON mode.
Gọi model giám khảo qua API: llama-3.1-8b-instant (https://api.groq.com/openai/v1)


/kaggle/working/MockProject_062026_NhomAI/Training/pipeline/evaluate.py:148: LangChainBetaWarning: Introduced in 0.2.24. API subject to change.
  rate_limiter = InMemoryRateLimiter(


Đang load embedding model (cho Answer Relevance)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

USE_RAG=False -> bỏ qua faithfulness/context_precision/context_recall (không dùng RAG, không có contexts). Chỉ chấm answer_relevancy_fast.
Tìm thấy 585 câu đã chấm từ lần chạy trước trong /kaggle/working/MockProject_062026_NhomAI/Training/output/ragas_scores.csv -> bỏ qua, chỉ chấm tiếp phần còn lại.
Còn 0/330 câu cần chấm (batch size = 5).
Không còn câu nào cần chấm -> dùng luôn kết quả đã có.
KẾT QUẢ ĐÁNH GIÁ (RAGAs + LLM Judge)
Tổng số câu đã chấm: 585
[CẢNH BÁO] Cột 'answer_relevancy': 6/585 câu (1.0%) bị NaN -- không chấm điểm được.
answer_relevancy    0.674396
dtype: float64

Chi tiết đầy đủ: /kaggle/working/MockProject_062026_NhomAI/Training/output/ragas_scores.csv


## Buoc 4: CHAT -- doc lap, tu chon bat/tat RAG rieng (khong lien quan co USE_RAG o Buoc 4)

Trong luc chat, go:
- `/rag on`  -> bat RAG cho cac cau hoi tiep theo
- `/rag off` -> tat RAG (Q&A thuan)
- `exit` hoac `quit` -> thoat

Neu USE_RAG cua ban chua bat (Buoc 4 dang False) ma muon thu RAG o day,
can dam bao da co chroma_db + da patch trust_remote_code (Buoc 2).

In [16]:
from pipeline import chat
chat.chat_loop()

Đang load model...
Load model xong.

CHATBOT SẴN SÀNG (mặc định USE_RAG=False)
Lệnh: '/rag on' bật RAG | '/rag off' tắt RAG | 'exit'/'quit' thoát



Câu hỏi của bạn:  /rag on


[OK] Đã BẬT RAG cho các câu hỏi tiếp theo.



Câu hỏi của bạn:  How to take care of a patient with heart disease


[rag_bridge] Đã load retrieval module từ: /kaggle/working/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5
[rag_bridge] RETRIEVAL_MODE = hybrid | SIMILARITY_THRESHOLD = 0.7


...


TRẢ LỜI: Proper hydration is important for heart health and should be monitored regularly.



Câu hỏi của bạn:  How to take care of a patient with heart disease



TRẢ LỜI: Take your medication as prescribed and discuss any side effects or concerns you have.



Câu hỏi của bạn:  exit


Thoát chatbot.
